## Homework

The goal of this homework is to familiarize users with monitoring for ML batch services, using PostgreSQL database to store metrics and Grafana to visualize them.

## Q1. Prepare the dataset

Start with `baseline_model_nyc_taxi_data.ipynb`. Download the March 2024 Green Taxi data. We will use this data to simulate a production usage of a taxi trip duration prediction service.

What is the shape of the downloaded data? How many rows are there?

* 72044
* 78537 
* 57457
* 54396

In [1]:
import os
import datetime

import requests
import pandas as pd
from joblib import dump

from evidently import ColumnMapping
from evidently.report import Report
from evidently.metrics import (
    ColumnDriftMetric, DatasetDriftMetric, DatasetMissingValuesMetric, ColumnQuantileMetric,
)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)


def download(file, folder='.'):
    """Download a NYC taxi file unless it's already there."""
    path = os.path.join(folder, file)
    if not os.path.exists(path):
        resp = requests.get(f'https://d37ci6vzurychx.cloudfront.net/trip-data/{file}')
        resp.raise_for_status()
        with open(path, 'wb') as f_out:
            f_out.write(resp.content)
    return path

In [2]:
march_data = pd.read_parquet(download('green_tripdata_2024-03.parquet'))
march_data.shape

(57457, 20)

**Answer:** 57457

## Baseline model and reference data

In [3]:
jan_data = pd.read_parquet(download('green_tripdata_2024-01.parquet', 'data'))

# target: trip duration in minutes
jan_data['duration_min'] = (jan_data.lpep_dropoff_datetime - jan_data.lpep_pickup_datetime).dt.total_seconds() / 60

# filter out outliers
jan_data = jan_data[(jan_data.duration_min >= 0) & (jan_data.duration_min <= 60)]
jan_data = jan_data[(jan_data.passenger_count > 0) & (jan_data.passenger_count <= 8)]

target = 'duration_min'
num_features = ['passenger_count', 'trip_distance', 'fare_amount', 'total_amount']
cat_features = ['PULocationID', 'DOLocationID']

train_data = jan_data[:30000].copy()
val_data = jan_data[30000:].copy()

model = LinearRegression()
model.fit(train_data[num_features + cat_features], train_data[target])

train_data['prediction'] = model.predict(train_data[num_features + cat_features])
val_data['prediction'] = model.predict(val_data[num_features + cat_features])

print('train MAE:', mean_absolute_error(train_data[target], train_data.prediction))
print('val MAE:  ', mean_absolute_error(val_data[target], val_data.prediction))

dump(model, 'models/lin_reg.bin')
val_data.to_parquet('data/reference.parquet')

train MAE: 3.5166246872282936
val MAE:   5.096439257058157


## Q2. Metric

Let's expand the number of data quality metrics we’d like to monitor! Please add one metric of your choice and a quantile value for the `"fare_amount"` column (`quantile=0.5`).

Hint: explore evidently metric `ColumnQuantileMetric` (from `evidently.metrics import ColumnQuantileMetric`) 

What metric did you choose?

In [4]:
column_mapping = ColumnMapping(
    target=None,
    prediction='prediction',
    numerical_features=num_features,
    categorical_features=cat_features,
)

report = Report(metrics=[
    ColumnDriftMetric(column_name='prediction'),
    DatasetDriftMetric(),
    DatasetMissingValuesMetric(),
    ColumnQuantileMetric(column_name='fare_amount', quantile=0.5),
])

In [5]:
# try it on the whole of March 2024
current = march_data.copy()
current['prediction'] = model.predict(current[num_features + cat_features].fillna(0))

report.run(reference_data=val_data, current_data=current, column_mapping=column_mapping)
report.as_dict()['metrics'][3]['result']

{'column_name': 'fare_amount',
 'column_type': 'num',
 'quantile': 0.5,
 'current': {'value': np.float64(13.5)},
 'reference': {'value': np.float64(12.8)}}

**Answer:** ColumnQuantileMetric(column_name='fare_amount', quantile=0.5) (median fare)

## Q3. Monitoring

Let’s start monitoring. Run expanded monitoring for a new batch of data (March 2024). 

What is the maximum value of metric `quantile = 0.5` on the `"fare_amount"` column during March 2024 (calculated daily)?

* 10
* 12.5
* 14.2
* 14.8

In [6]:
daily = []
for i in range(31):
    start = datetime.datetime(2024, 3, 1) + datetime.timedelta(days=i)
    end = start + datetime.timedelta(days=1)
    day_data = current[(current.lpep_pickup_datetime >= start) & (current.lpep_pickup_datetime < end)]

    report.run(reference_data=val_data, current_data=day_data, column_mapping=column_mapping)
    daily.append({'day': start.date(), 'fare_amount_median': report.as_dict()['metrics'][3]['result']['current']['value']})

daily = pd.DataFrame(daily)
print('max daily median:', daily.fare_amount_median.max())
daily

max daily median: 14.2


,day,fare_amount_median
0,2024-03-01,13.5
1,2024-03-02,13.5
2,2024-03-03,14.2
3,2024-03-04,12.8
4,2024-03-05,13.5
5,2024-03-06,12.8
6,2024-03-07,13.5
7,2024-03-08,13.5
8,2024-03-09,13.5
9,2024-03-10,14.2


**Answer:** 14.2

### Store the daily metrics in PostgreSQL

In [7]:
!docker compose up -d

[+] up 2/5
 ✔ Network module5_back-tier   Created                                      0.0s
 ✔ Network module5_front-tier  Created                                      0.0s
 ⠋ Container module5-adminer-1 Creating                                     0.0s
 ⠋ Container module5-db-1      Creating                                     0.0s
 ⠋ Container module5-grafana-1 Creating                                     0.0s
[+] up 5/5


 ✔ Container module5-adminer-1 Created                                      0.1s
 ✔ Container module5-db-1      Created                                      0.1s
 ✔ Container module5-grafana-1 Created                                      0.1s
[+] up 2/5


 ⠹ Container module5-adminer-1 Starting                                     0.2s
 ⠹ Container module5-db-1      Starting                                     0.2s
 ⠹ Container module5-grafana-1 Starting                                     0.2s
[+] up 3/5


 ⠸ Container module5-adminer-1 Starting                    

In [9]:
%run evidently_metrics_calculation.py

INFO:prefect.flow_runs:Beginning flow run 'tall-hare' for flow 'batch-monitoring'
INFO:prefect.flow_runs:View at http://127.0.0.1:4200/runs/flow-run/344649e3-f77e-430c-b89c-6ec66401f873
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


2024-03-01  fare_amount median = 13.5


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


2024-03-02  fare_amount median = 13.5


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


...

INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


2024-03-30  fare_amount median = 14.2


INFO:prefect.task_runs:Finished in state Completed()
INFO:prefect.task_runs:Finished in state Completed()


2024-03-31  fare_amount median = 13.5


INFO:prefect.flow_runs:Finished in state Completed()


## Q4. Dashboard


Finally, let’s add panels with new added metrics to the dashboard. After we customize the  dashboard let's save a dashboard config, so that we can access it later. Hint: click on “Save dashboard” to access JSON configuration of the dashboard. This configuration should be saved locally.

Where to place a dashboard config file?

* `project_folder` (05-monitoring)
* `project_folder/config`  (05-monitoring/config)
* `project_folder/dashboards`  (05-monitoring/dashboards)
* `project_folder/data`  (05-monitoring/data)

**Answer:** project_folder/dashboards

In [10]:
!docker compose down

[+] down 0/3
 ⠋ Container module5-adminer-1 Stopping                                     0.1s
 ⠋ Container module5-grafana-1 Stopping                                     0.1s
 ⠋ Container module5-db-1      Stopping                                     0.1s

 ⠙ Container module5-adminer-1 Stopping                                     0.2s
 ⠙ Container module5-grafana-1 Stopping                                     0.2s
 ⠙ Container module5-db-1      Stopping                                     0.2s

 ⠹ Container module5-adminer-1 Stopping                                     0.3s
 ⠹ Container module5-grafana-1 Stopping                                     0.3s
 ⠹ Container module5-db-1      Stopping                                     0.3s
[+] down 2/3
 ✔ Container module5-adminer-1 Removed                                      0.3s
 ⠸ Container module5-grafana-1 Stopping                                     0.4s
 ✔ Container module5-db-1      Removed                                      0.3s


## Submit the results

* Submit your answers here: https://courses.datatalks.club/mlops-zoomcamp-2025/homework/hw5